# Generate COMPASS platinum-arm figures — R port

R implementation of `COMPASS_generate_figures.ipynb` using tidyverse/ggplot2.
**Run All** renders the six ARPI-restricted cohort arms: ICD-C61,
VTE-project prostate, and their union, each with and without the
non-prostate-primary exclusion.

Outputs are organized by figure, then panel, for direct cohort comparison:

`FIG_ROOT/<figure>/<plot-stem>/R/<cohort>_<plot-stem>.png`

For example, `figure1/figure1a_consort/R/` contains the Figure 1A PNG from all
six cohorts. Plot PDFs are not generated. Tables remain CSV/Markdown and use
the same figure/panel/cohort-prefix convention.

All figure-generation logic lives in `COMPASS_generate_figures_pipeline.R` as
`generate_figures(cohort, ...)`. This notebook just sources that file and
calls it once per cohort, in the same R session, from a single cell.


## Setup


In [ ]:
pipeline_path <- normalizePath(
  file.path(getwd(), "COMPASS_generate_figures_pipeline.R"),
  mustWork = FALSE
)
if (!file.exists(pipeline_path)) {
  pipeline_path <- normalizePath(
    file.path(getwd(), "COMPASS", "survival_analysis", "COMPASS_generate_figures_pipeline.R"),
    mustWork = FALSE
  )
}
if (!file.exists(pipeline_path))
  stop("Could not locate COMPASS_generate_figures_pipeline.R")
source(pipeline_path)

NEPC_PROJ_PATH <- "/data/gusev/USERS/jpconnor/data/CAIA/COMPASS"
FIG_ROOT <- "/data/gusev/USERS/jpconnor/figures/CAIA/COMPASS"


## Render all cohorts

Runs the full figure-generation pipeline once per cohort, in the same R
session. Each call is self-contained (loads its own inputs, writes its own
cohort-prefixed outputs); a failure on one cohort is reported with its
cohort name rather than aborting the whole run silently.


In [ ]:
failures <- character(0)
for (cohort in COHORTS) {
  message(sprintf("\n========== R figures: %s ==========", cohort))
  result <- tryCatch({
    generate_figures(cohort, nepc_proj_path = NEPC_PROJ_PATH, fig_root = FIG_ROOT)
    TRUE
  }, error = function(e) {
    message(sprintf("FAILED %s: %s", cohort, conditionMessage(e)))
    FALSE
  })
  if (isTRUE(result)) {
    message(sprintf("completed %s", cohort))
  } else {
    failures <- c(failures, cohort)
  }
}

if (length(failures) > 0)
  stop(sprintf("R figure generation failed for cohorts: %s", paste(failures, collapse = ", ")))
message("\nAll R cohort figure sets completed.")
